# Tutorial 08 Text, Sentiment, and Word Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_08_Text_and_Sequences/tutorial_08_pytorch.ipynb)

Follow text through token IDs and embedding vectors, train a masked-pooling sentiment classifier, compare a TF-IDF baseline, and learn separate Skip-gram embeddings from movie reviews.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Python and supervised learning (Tutorials 01–03); no previous runtime or pretrained language model needed.

**Resources:** CPU supported. IMDb archive approximately 84 MB; quick mode uses 6,000/1,000/2,000 train/validation/test reviews, eight sentiment epochs, and three small Skip-gram epochs. The full archive still downloads. Allow 60–90 minutes for reading and exercises.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals and boundaries

1. Explain tokenization, arbitrary integer IDs, one-hot vectors, and learned embeddings.
2. Build vocabulary and TF-IDF features from training text only; distinguish padding from unknown tokens.
3. Train a binary classifier with masked mean pooling, select on validation data, and evaluate once on held-out test reviews.
4. Construct ordered target/context pairs and optimize Skip-gram with negative sampling (SGNS).
5. Inspect cosine neighbors and a two-dimensional projection without treating them as proof of semantic understanding; export **word strings** with vectors and reload a complete sentiment checkpoint.

The sentiment model learns embeddings from **review labels**. Word2Vec learns different embeddings from **local word co-occurrence**, without sentiment labels. We train these separately; neither is a pretrained transformer. Mean pooling loses order, and the SGNS context window also ignores relative position. Neither is an RNN or an order-aware sequence model.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps an existing PyTorch/torchvision pair together and does not downgrade Colab’s installed stack. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'torch': 'torch', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import re, html, json, time, platform, random, hashlib, tarfile, urllib.request
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,TensorDataset
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss,accuracy_score,confusion_matrix
from sklearn.decomposition import PCA
from IPython.display import display
%matplotlib inline
SEED=42
QUICK_RUN=True
DEVICE_PREFERENCE="auto"  # "cpu" forces CPU.
assert DEVICE_PREFERENCE in {"auto","cpu"}
device=torch.device("cuda" if DEVICE_PREFERENCE=="auto" and torch.cuda.is_available() else "cpu")
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
MAX_VOCAB=8000
MAX_LENGTH=200
EMBED_DIM=32
EPOCHS=8 if QUICK_RUN else 15
SG_EPOCHS=3 if QUICK_RUN else 8
DATA_DIR=Path("data/tutorial_08");OUTPUT_DIR=Path("outputs/tutorial_08")
DATA_DIR.mkdir(parents=True,exist_ok=True);OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
versions={"python":platform.python_version(),"torch":str(torch.__version__),"numpy":np.__version__,
    "pandas":pd.__version__,"matplotlib":matplotlib.__version__,"sklearn":sklearn.__version__,
    "device":str(device),"seed":SEED,"quick_run":QUICK_RUN}
display(pd.Series(versions,name="Environment"))
run_started=time.perf_counter()

## 2. Download IMDb and split before fitting anything

IMDb provides separate labelled training and test directories. We take validation reviews from the training directory and keep the original test directory separate. Only `pos` and `neg` are loaded; no need to delete `unsup`. Label 1 means positive, label 0 negative. The source contains strongly polarized reviews rather than a representative sample of all opinions.

Quick mode chooses balanced deterministic subsets. Larger mode uses 20,000/5,000/25,000 reviews. Archive SHA-256 records downloaded bytes, not independent authenticity. We also remove exact raw-text duplicates across partitions, retaining training first and validation second; resulting counts may be slightly smaller. This does not detect every near-duplicate or movie-level relationship.

In [ ]:
DATA_URL="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
archive_path=DATA_DIR/"aclImdb_v1.tar.gz"
if not archive_path.exists():
    print("Downloading IMDb (approximately 84 MB)...",flush=True)
    partial=archive_path.with_suffix(".part")
    with urllib.request.urlopen(DATA_URL,timeout=180) as response:
        with partial.open("wb") as output:
            while chunk:=response.read(1024*1024):output.write(chunk)
    partial.replace(archive_path)
archive_sha256=hashlib.sha256(archive_path.read_bytes()).hexdigest()
root=DATA_DIR.resolve()
# Only extract labelled text plus dataset documentation; never delete the source data.
with tarfile.open(archive_path,"r:gz") as archive:
    members=[m for m in archive.getmembers() if m.isfile() and
        (re.fullmatch(r"aclImdb/(train|test)/(pos|neg)/[^/]+\.txt",m.name) or m.name=="aclImdb/README")]
    assert members
    for member in members:
        assert (root/member.name).resolve().is_relative_to(root)
    archive.extractall(root,members=members,filter="data")
corpus_root=DATA_DIR/"aclImdb"
rng=np.random.default_rng(SEED)
train_records=[];val_records=[];test_records=[]
for label,name in [(0,"neg"),(1,"pos")]:
    source=sorted((corpus_root/"train"/name).glob("*.txt"))
    tests=sorted((corpus_root/"test"/name).glob("*.txt"))
    assert len(source)==12500 and len(tests)==12500
    source=[source[i] for i in rng.permutation(len(source))]
    tests=[tests[i] for i in rng.permutation(len(tests))]
    ntrain,nval,ntest=(3000,500,1000) if QUICK_RUN else (10000,2500,12500)
    for group,files in [(train_records,source[:ntrain]),(val_records,source[ntrain:ntrain+nval]),(test_records,tests[:ntest])]:
        group.extend((p.read_text(encoding="utf-8"),label,str(p.relative_to(corpus_root))) for p in files)
seen=set();dedup_removed={}
for name,group in [("train",train_records),("validation",val_records),("test",test_records)]:
    kept=[]
    for record in group:
        digest=hashlib.sha256(record[0].encode()).hexdigest()
        if digest not in seen:kept.append(record);seen.add(digest)
    dedup_removed[name]=len(group)-len(kept);group[:]=kept
    rng.shuffle(group)
sets=[{r[2] for r in group} for group in [train_records,val_records,test_records]]
assert not sets[0]&sets[1] and not sets[0]&sets[2] and not sets[1]&sets[2]
train_text=[r[0] for r in train_records];val_text=[r[0] for r in val_records];test_text=[r[0] for r in test_records]
y_train=np.array([r[1] for r in train_records]);y_val=np.array([r[1] for r in val_records]);y_test=np.array([r[1] for r in test_records])
split_summary={name:{"reviews":len(group),"positive":sum(r[1] for r in group)}
    for name,group in [("train",train_records),("validation",val_records),("test",test_records)]}
display(pd.DataFrame(split_summary).T);print("Exact-text duplicates removed:",dedup_removed)
print("Archive SHA-256:",archive_sha256)

## 3. Text → tokens → IDs → embedding vectors

Our simple English tokenizer lowercases, removes HTML tags, decodes HTML entities, and keeps alphabetic words with internal apostrophes. It is a teaching choice, not a universal tokenizer: punctuation, numbers, and some linguistic information are lost. Negation words are retained.

Vocabulary frequency is computed **only from training text**, with deterministic alphabetical tie-breaking. ID 0 is `<pad>`, ID 1 `<unk>`. Numerical closeness of IDs has no semantic meaning. One-hot multiplication and an embedding lookup give the same result; the latter avoids allocating a large sparse-looking dense matrix.

In [ ]:
TOKEN_PATTERN=r"[a-z]+(?:'[a-z]+)?"
def tokenize(text):
    return re.findall(TOKEN_PATTERN,html.unescape(re.sub(r"<[^>]+>"," ",text)).lower())
train_tokens=[tokenize(text) for text in train_text]
counts=Counter(token for tokens in train_tokens for token in tokens)
words=[word for word,count in sorted(counts.items(),key=lambda pair:(-pair[1],pair[0]))[:MAX_VOCAB-2]]
vocab={word:i for i,word in enumerate(["<pad>","<unk>"]+words)}
itos=list(vocab)
assert all(vocab[word]==i for i,word in enumerate(itos))
def encode(text,max_length=MAX_LENGTH,mapping=None):
    mapping=vocab if mapping is None else mapping
    ids=[mapping.get(word,1) for word in tokenize(text)[:max_length]]
    return ids or [1]  # Empty text becomes one unknown token, never a divide-by-zero case.

demo="The movie was not good, but the acting was excellent."
demo_tokens=tokenize(demo);demo_ids=torch.tensor(encode(demo),dtype=torch.long)
lookup=nn.Embedding(len(vocab),6,padding_idx=0)
with torch.no_grad():
    demo_vectors=lookup(demo_ids)
    one_hot=F.one_hot(demo_ids,num_classes=len(vocab)).float()
    torch.testing.assert_close(one_hot@lookup.weight,demo_vectors)
display(pd.DataFrame({"token":demo_tokens,"id":demo_ids.tolist()}))
fig,axes=plt.subplots(1,2,figsize=(12,4),layout="constrained")
axes[0].imshow(demo_ids.numpy()[:,None],aspect="auto",cmap="Blues")
axes[0].set(yticks=range(len(demo_tokens)),yticklabels=demo_tokens,xticks=[0],xticklabels=["arbitrary ID"],title="Tokens mapped to IDs")
for i,value in enumerate(demo_ids):axes[0].text(0,i,str(value.item()),ha="center",va="center",color="white" if value.item()>demo_ids.max().item()/2 else "black")
plot=axes[1].imshow(demo_vectors.numpy(),aspect="auto",cmap="coolwarm")
axes[1].set(yticks=range(len(demo_tokens)),yticklabels=demo_tokens,xlabel="Embedding dimension",title="Random lookup vectors (not yet trained)")
fig.colorbar(plot,ax=axes[1]);fig.savefig(OUTPUT_DIR/"token_to_embedding.png",dpi=140);plt.show()
print("Vocabulary size:",len(vocab),"one-hot shape:",tuple(one_hot.shape),"embedding shape:",tuple(demo_vectors.shape))

In [ ]:
def encode_dataset(texts,labels):
    rows=[encode(text) for text in texts]
    matrix=torch.zeros((len(rows),MAX_LENGTH),dtype=torch.long)
    for i,row in enumerate(rows):matrix[i,:len(row)]=torch.tensor(row)
    return TensorDataset(matrix,torch.tensor(labels,dtype=torch.float32))
train_data=encode_dataset(train_text,y_train);val_data=encode_dataset(val_text,y_val)
def loader(dataset,shuffle=False):
    return DataLoader(dataset,batch_size=128,shuffle=shuffle,num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

def token_audit(texts):
    tokens=[tokenize(text) for text in texts]
    retained=[t for row in tokens for t in row[:MAX_LENGTH]]
    return {"truncated_reviews":sum(len(row)>MAX_LENGTH for row in tokens),
            "unknown_fraction":sum(t not in vocab for t in retained)/max(1,len(retained))}
display(pd.DataFrame({"train":token_audit(train_text),"validation":token_audit(val_text)}).T)
print("Input batch shape:",next(iter(loader(train_data)))[0].shape)

## 4. Padding-aware sentiment classification

Setting `padding_idx=0` prevents the pad embedding from learning, but ordinary average pooling still divides by padded sequence length. We explicitly sum non-padding vectors and divide by the number of real tokens. The assertion below checks that appending padding does not change predictions.

The classifier emits one raw logit per review. `BCEWithLogitsLoss` expects logits; prediction uses logit ≥ 0 (sigmoid ≥ 0.5). This model cannot distinguish permutations of the same retained words; we test that limitation too. Empty input has a defined numeric path, but its score is not a meaningful sentiment judgment.

In [ ]:
class SentimentModel(nn.Module):
    def __init__(self,size,dimension=EMBED_DIM):
        super().__init__()
        self.embedding=nn.Embedding(size,dimension,padding_idx=0)
        self.head=nn.Sequential(nn.Linear(dimension,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,ids):
        mask=(ids!=0).unsqueeze(-1)
        pooled=(self.embedding(ids)*mask).sum(1)/mask.sum(1).clamp_min(1)
        return self.head(pooled).squeeze(-1)
torch.manual_seed(SEED)
model=SentimentModel(len(vocab)).to(device)
sample=torch.tensor([encode("this movie was excellent")],device=device)
with torch.no_grad():
    torch.testing.assert_close(model(sample),model(F.pad(sample,(0,7))),rtol=1e-5,atol=1e-6)
    torch.testing.assert_close(model(sample),model(sample.flip(1)),rtol=1e-5,atol=1e-6)
    assert torch.isfinite(model(torch.tensor([encode("")],device=device))).all()
print("Padding invariance, permutation invariance, and empty-input checks passed.")

def sentiment_epoch(dataset,optimizer=None):
    model.train(optimizer is not None)
    loss_sum=correct=number=0
    with torch.set_grad_enabled(optimizer is not None):
        for ids,labels in loader(dataset,shuffle=optimizer is not None):
            ids,labels=ids.to(device),labels.to(device)
            logits=model(ids);loss=F.binary_cross_entropy_with_logits(logits,labels)
            assert torch.isfinite(loss)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
            loss_sum+=loss.item()*len(labels);correct+=((logits>=0)==labels.bool()).sum().item();number+=len(labels)
    return {"loss":loss_sum/number,"accuracy":correct/number}
optimizer=torch.optim.Adam(model.parameters(),lr=0.003)
best_loss=float("inf");best_epoch=0;history=[]
for epoch in range(1,EPOCHS+1):
    training=sentiment_epoch(train_data,optimizer);validation=sentiment_epoch(val_data)
    history.append({"epoch":epoch,"train_loss":training["loss"],"val_loss":validation["loss"],
        "train_accuracy":training["accuracy"],"val_accuracy":validation["accuracy"]})
    if validation["loss"]<best_loss:
        best_loss=validation["loss"];best_epoch=epoch
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    print(f"Epoch {epoch}: train {training['accuracy']:.1%}, validation {validation['accuracy']:.1%}")
model.load_state_dict(best_state);model.eval()
assert torch.count_nonzero(model.embedding.weight[0])==0
history=pd.DataFrame(history)
fig,axes=plt.subplots(1,2,figsize=(11,3.6),layout="constrained")
for prefix,title in [("train","Training"),("val","Validation")]:
    axes[0].plot(history.epoch,history[prefix+"_loss"],label=title)
    axes[1].plot(history.epoch,history[prefix+"_accuracy"],label=title)
axes[0].set(ylabel="Binary cross-entropy",xlabel="Epoch");axes[1].set(ylabel="Accuracy",xlabel="Epoch",ylim=(0,1))
for ax in axes:ax.axvline(best_epoch,color="gray",ls="--");ax.legend()
fig.savefig(OUTPUT_DIR/"sentiment_training.png",dpi=140);plt.show()
print("Restored validation-selected epoch:",best_epoch)

## 5. Bag-of-words baseline and held-out evaluation

Compare the neural model against TF-IDF plus logistic regression and a training-majority baseline. TF-IDF weights document word counts by corpus frequency; it also discards word order. Fit vocabulary/IDF only on training data. Both learned classifiers see the same first 200 tokens per review so input truncation does not silently advantage one model.

Choose the model with lower validation log loss, then report both prespecified models on the held-out original test partition. Do not tune after seeing test results. The quick run is an instructional subset, not a leaderboard result. Baseline performance can exceed a neural model.

In [ ]:
def retained_text(text):return " ".join(tokenize(text)[:MAX_LENGTH])
tfidf=TfidfVectorizer(tokenizer=str.split,token_pattern=None,lowercase=False,max_features=MAX_VOCAB-2)
x_train=tfidf.fit_transform([retained_text(text) for text in train_text])
x_val=tfidf.transform([retained_text(text) for text in val_text])
bow_model=LogisticRegression(C=1.0,max_iter=300,random_state=SEED)
bow_model.fit(x_train,y_train)
@torch.no_grad()
def probabilities(dataset):
    model.eval()
    return torch.cat([model(ids.to(device)).sigmoid().cpu() for ids,_ in loader(dataset)]).numpy()
def score(labels,p):return {"log_loss":float(log_loss(labels,p,labels=[0,1])),"accuracy":float(accuracy_score(labels,p>=0.5))}
val_prob={"Masked embeddings":probabilities(val_data),"TF-IDF":bow_model.predict_proba(x_val)[:,1]}
validation_results={name:score(y_val,p) for name,p in val_prob.items()}
selected=min(validation_results,key=lambda name:validation_results[name]["log_loss"])
display(pd.DataFrame(validation_results).T);print("Selected by validation:",selected)
# No test text enters vocabulary, IDF fitting, optimizer steps, or model selection.
test_data=encode_dataset(test_text,y_test)
x_test=tfidf.transform([retained_text(text) for text in test_text])
test_prob={"Masked embeddings":probabilities(test_data),"TF-IDF":bow_model.predict_proba(x_test)[:,1]}
test_results={name:score(y_test,p) for name,p in test_prob.items()}
majority=int(y_train.mean()>=0.5);baseline=float((y_test==majority).mean())
display(pd.DataFrame(test_results).T);print("Training-majority test accuracy:",baseline)
fig,axes=plt.subplots(1,2,figsize=(10,4),layout="constrained")
axes[0].bar(list(test_results),[r["accuracy"] for r in test_results.values()])
axes[0].axhline(baseline,color="black",ls="--",label="Training-majority baseline");axes[0].legend(fontsize=8)
axes[0].set(ylabel="Test accuracy",ylim=(0,1),title="Prespecified models")
cm=confusion_matrix(y_test,test_prob[selected]>=0.5,labels=[0,1])
axes[1].imshow(cm,cmap="Blues");axes[1].set(xticks=[0,1],yticks=[0,1],xticklabels=["negative","positive"],yticklabels=["negative","positive"],xlabel="Predicted",ylabel="True",title=f"Validation-selected: {selected}")
for i in range(2):
    for j in range(2):axes[1].text(j,i,str(cm[i,j]),ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
fig.savefig(OUTPUT_DIR/"sentiment_test.png",dpi=140);plt.show()

## 6. Skip-gram: learn from neighboring words

Now switch objectives. A target token predicts nearby context words without review labels. For radius 2, context includes up to two tokens on either side within the same review. We demonstrate pair positions on a sentence, then build pairs from many **training** reviews. This implementation does not implement frequent-word subsampling, phrase discovery, or dynamic windows.

Negative sampling draws noise words with probability proportional to training frequency raised to 0.75. PAD and UNK have zero probability. Sampling is with replacement and uses an advancing RNG: a noise word may coincide with a true context or another noise word. These are sampled contrastive examples, not guaranteed semantic negatives. SGNS is a binary logistic objective, not normalized next-word probabilities or an exact softmax/NCE loss.

In [ ]:
toy=tokenize("The wide road shimmered in the hot sun.")
def pair_positions(length,radius=2):
    return [(i,j) for i in range(length) for j in range(max(0,i-radius),min(length,i+radius+1)) if i!=j]
toy_positions=pair_positions(len(toy))
assert len(toy_positions)==26 and all(i!=j and abs(i-j)<=2 for i,j in toy_positions)
display(pd.DataFrame([(i,toy[i],j,toy[j]) for i,j in toy_positions],columns=["target position","target","context position","context"]).head(12))
corpus_limit=1000 if QUICK_RUN else len(train_tokens)
pairs=[]
for tokens in train_tokens[:corpus_limit]:
    ids=[vocab.get(token,1) for token in tokens[:80]]
    pairs.extend((ids[i],ids[j]) for i,j in pair_positions(len(ids)) if ids[i]>=2 and ids[j]>=2)
pairs=np.asarray(pairs,dtype=np.int64)
limit=40000 if QUICK_RUN else 300000
rng=np.random.default_rng(SEED)
if len(pairs)>limit:pairs=pairs[rng.choice(len(pairs),limit,replace=False)]
assert len(pairs)>1000 and pairs.min()>=2
noise=torch.tensor([0.,0.]+[counts[word]**0.75 for word in itos[2:]],dtype=torch.float32)
noise/=noise.sum()
noise_generator=torch.Generator().manual_seed(SEED)
def sample_noise(batch_size,k=5):
    return torch.multinomial(noise,batch_size*k,replacement=True,generator=noise_generator).reshape(batch_size,k)
noise_demo=sample_noise(4)
assert (noise_demo>=2).all()
display(pd.DataFrame([[itos[i] for i in row] for row in noise_demo.tolist()]))
print("Training reviews used for pair candidates:",min(corpus_limit,len(train_tokens)),"sampled pairs:",len(pairs))

In [ ]:
class SkipGram(nn.Module):
    def __init__(self,size,dimension=32):
        super().__init__()
        self.target=nn.Embedding(size,dimension,padding_idx=0)
        self.context=nn.Embedding(size,dimension,padding_idx=0)
        nn.init.uniform_(self.target.weight,-0.5/dimension,0.5/dimension)
        nn.init.zeros_(self.context.weight)
        with torch.no_grad():self.target.weight[0].zero_()
    def loss(self,targets,contexts,negatives):
        target=self.target(targets)
        positive=(target*self.context(contexts)).sum(1)
        negative=torch.bmm(self.context(negatives),target.unsqueeze(2)).squeeze(2)
        return (-F.logsigmoid(positive)-F.logsigmoid(-negative).sum(1)).mean()
torch.manual_seed(SEED)
sg=SkipGram(len(vocab)).to(device)
sg_optimizer=torch.optim.Adam(sg.parameters(),lr=0.01)
pair_data=TensorDataset(torch.from_numpy(pairs[:,0]),torch.from_numpy(pairs[:,1]))
sg_loader=DataLoader(pair_data,batch_size=1024,shuffle=True,num_workers=0,generator=torch.Generator().manual_seed(SEED))
sg_history=[]
for epoch in range(1,SG_EPOCHS+1):
    total=count=0
    for target,context in sg_loader:
        negatives=sample_noise(len(target)).to(device)
        loss=sg.loss(target.to(device),context.to(device),negatives)
        assert torch.isfinite(loss)
        sg_optimizer.zero_grad(set_to_none=True);loss.backward();sg_optimizer.step()
        total+=loss.item()*len(target);count+=len(target)
    sg_history.append(total/count)
    print(f"SGNS epoch {epoch}: mean loss per positive pair = {total/count:.4f}")
assert np.isfinite(sg_history).all()
fig,ax=plt.subplots(figsize=(6,3),layout="constrained")
ax.plot(range(1,SG_EPOCHS+1),sg_history,marker="o");ax.set(xlabel="Epoch",ylabel="SGNS loss per positive pair",title="Five noise samples per pair; not sentiment accuracy")
fig.savefig(OUTPUT_DIR/"skipgram_loss.png",dpi=140);plt.show()

## 7. Inspect embeddings without over-interpreting them

Cosine neighbors below come from the full 32-dimensional vectors; PCA is only a two-dimensional projection of a selected group of frequent words. Axis signs, distances, and apparent clusters in the projection are not a semantic evaluation. A small corpus and a few epochs can produce poor neighbors. Distributional similarity can group antonyms because they occur in similar contexts.

We show separate sentiment-trained and SGNS embeddings. SGNS vectors for words absent from positive pairs may be untrained, so inspect only vocabulary IDs that occur as targets. No claims about analogy solving or general-purpose language understanding follow from a decreasing training loss.

In [ ]:
sentiment_vectors=model.embedding.weight.detach().cpu().numpy()
sg_vectors=sg.target.weight.detach().cpu().numpy()
observed_ids=np.unique(pairs[:,0]);observed_set=set(observed_ids.tolist())
def neighbors(matrix,word,k=5):
    if word not in vocab or vocab[word] not in observed_set:return []
    normalized=matrix/np.maximum(np.linalg.norm(matrix,axis=1,keepdims=True),1e-12)
    scores=normalized[observed_ids]@normalized[vocab[word]]
    order=np.argsort(-scores)
    return [(itos[observed_ids[i]],round(float(scores[i]),3)) for i in order if observed_ids[i]!=vocab[word]][:k]
for word in ["good","bad","movie","excellent"]:
    print(word,"sentiment:",neighbors(sentiment_vectors,word),"SGNS:",neighbors(sg_vectors,word))
plot_ids=observed_ids[:12]
fig,axes=plt.subplots(1,2,figsize=(12,5),layout="constrained")
for ax,matrix,title in zip(axes,[sentiment_vectors,sg_vectors],["Sentiment-trained embeddings","Skip-gram embeddings"]):
    projection=PCA(n_components=2).fit_transform(matrix[plot_ids])
    ax.scatter(projection[:,0],projection[:,1],s=20)
    for j,(point,idx) in enumerate(zip(projection,plot_ids)):
        ax.annotate(itos[idx],point,fontsize=9,xytext=(-28,12) if j%2 else (6,-16),textcoords="offset points",arrowprops={"arrowstyle":"-","color":"gray","lw":0.5})
    ax.margins(0.2);ax.set(title=title,xlabel="PC1",ylabel="PC2")
fig.savefig(OUTPUT_DIR/"embedding_projection.png",dpi=140);plt.show()

## 8. Export actual words and reload the sentiment model

A vectors TSV and metadata TSV must have matching rows. Export word strings, not dictionary keys accidentally interpreted as numeric IDs. We omit PAD/UNK and restrict SGNS export to observed target words. TSV files can be used in an embedding viewer; no online viewer is needed to finish the notebook.

The sentiment checkpoint includes model weights, the exact vocabulary, tokenizer configuration, max length, and label mapping. It saves the neural model even if TF-IDF won validation. The checkpoint is for inference with `SentimentModel` and the documented tokenizer; it is not the selected baseline's artifact. Rebuild and reload on CPU, then compare raw logits.

In [ ]:
export_rows={}
for prefix,matrix,indices in [("sentiment",sentiment_vectors,np.arange(2,len(vocab))),("skipgram",sg_vectors,observed_ids)]:
    np.savetxt(OUTPUT_DIR/f"{prefix}_vectors.tsv",matrix[indices],delimiter="\t",fmt="%.8g")
    (OUTPUT_DIR/f"{prefix}_metadata.tsv").write_text("\n".join(itos[i] for i in indices)+"\n",encoding="utf-8")
    metadata=(OUTPUT_DIR/f"{prefix}_metadata.tsv").read_text().splitlines()
    loaded_vectors=np.loadtxt(OUTPUT_DIR/f"{prefix}_vectors.tsv",delimiter="\t")
    assert metadata==[itos[i] for i in indices] and len(metadata)==len(loaded_vectors)
    np.testing.assert_allclose(loaded_vectors,matrix[indices],rtol=1e-6,atol=1e-7)
    export_rows[prefix]=len(indices)
checkpoint={"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},"vocab":vocab,
    "embedding_dim":EMBED_DIM,"max_length":MAX_LENGTH,"token_pattern":TOKEN_PATTERN,
    "preprocessing":"strip HTML tags, html.unescape, lowercase, regex findall; empty -> UNK",
    "labels":{"negative":0,"positive":1},"best_epoch":best_epoch}
torch.save(checkpoint,OUTPUT_DIR/"sentiment_model.pt")
saved=torch.load(OUTPUT_DIR/"sentiment_model.pt",map_location="cpu",weights_only=True)
restored=SentimentModel(len(saved["vocab"]),saved["embedding_dim"]).eval()
restored.load_state_dict(saved["state_dict"])
probes=["The acting was excellent.","I did not like this movie.",""]
probe_ids=[encode(text,max_length=saved["max_length"],mapping=saved["vocab"]) for text in probes]
probe_tensor=nn.utils.rnn.pad_sequence([torch.tensor(row) for row in probe_ids],batch_first=True,padding_value=0)
with torch.no_grad():
    original=model(probe_tensor.to(device)).cpu();replayed=restored(probe_tensor)
    torch.testing.assert_close(original,replayed,rtol=1e-4,atol=1e-5)
    torch.testing.assert_close(replayed,restored(F.pad(probe_tensor,(0,11))),rtol=1e-5,atol=1e-6)
print("Word/vector alignment, TSV round trips, trained-model padding invariance, CPU checkpoint reload: passed.")
report={"environment":versions,"data_url":DATA_URL,"archive_sha256":archive_sha256,
    "split_summary":split_summary,"dedup_removed":dedup_removed,"vocab_size":len(vocab),"max_length":MAX_LENGTH,
    "sentiment_epochs":EPOCHS,"best_epoch":best_epoch,"validation":validation_results,"test":test_results,
    "selected_by_validation":selected,"baseline_accuracy":baseline,"skipgram":{"pairs":len(pairs),
    "training_reviews":min(corpus_limit,len(train_tokens)),"epochs":SG_EPOCHS,"noise_samples":5,"losses":sg_history},
    "export_rows":export_rows,"checks":{"one_hot_lookup":True,"padding_invariance":True,"empty_input":True,
    "disjoint_paths":True,"exact_text_dedup":True,"finite_losses":True,"tsv_alignment":True,"checkpoint_roundtrip":True},
    "seconds_after_imports":time.perf_counter()-run_started}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(report,indent=2,allow_nan=False))
print("TUTORIAL_08_COMPLETE");print(json.dumps(report,indent=2,allow_nan=False))

## Exercises

1. Compare the token IDs for `good` and `bad`: why does their numeric difference tell you nothing about meaning?
2. Remove masked division temporarily and test appending padding. Explain what changes, then restore the correct code.
3. Compare two permutations of the same tokens. What additional architecture would be needed to use order?
4. Change one SGNS setting (window radius, embedding dimension, or number of noise samples). State the training objective and compute budget before comparing neighbors.
5. Discuss why a TF-IDF baseline can win, why test data must not build the vocabulary, and why the classifier is not suitable for deciding whether a person is trustworthy or truthful.

Default outputs include five figures, aligned vector/word TSVs, a sentiment checkpoint, and `metrics.json`. Download them before the Colab runtime disconnects.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_08/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

License information is available in the repository `LICENSE`.

References: [IMDb / Maas et al. dataset](https://ai.stanford.edu/~amaas/data/sentiment/), [PyTorch Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [Mikolov et al. negative sampling paper](https://arxiv.org/abs/1310.4546). Dataset terms and attribution are separate from the course code license; no reviews or pretrained weights are committed here.

Compared with the source lesson, this version fixes padding-sensitive pooling, index-only metadata exports, repeatedly reseeded negative sampling, and misleading single-sentence Word2Vec claims; adds actual held-out evaluation, plots, baseline comparison, and reload checks. No TensorFlow, torchtext, API key, or pretrained language model is required.